# Chapter 5: AI-Assisted Requirements & Planning

This notebook provides a runnable companion to Chapter 5 of the SDLC book. It follows the **GlobalBank Corporate Onboarding** case study, demonstrating how AI can be integrated into every stage of the requirements and planning pipeline.

### Prerequisites
To run this notebook, you will need an OpenAI API key. We will use the `langchain-openai` package to structure our prompts and interactions.

In [ ]:
!pip install -q langchain langchain-openai pydantic

import os
import getpass

# Set your OpenAI API Key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

## 1. Stakeholder Discovery (§5.1)
Before eliciting requirements, we need to know who to talk to. We'll use AI to analyze the project description and identify hidden stakeholders.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

project_description = """
GlobalBank Onboarding is a new digital platform for onboarding corporate clients. 
It needs to automate entity resolution for complex corporate structures, perform UK FCA 
KYC/AML screening in real-time, generate risk decisioning scores, and provision 
accounts in the core banking system.
"""

stakeholder_prompt = PromptTemplate.from_template("""
You are a senior business analyst. Given the following project description,
identify 5 stakeholders who should be consulted during requirements elicitation.

For each stakeholder provide:
- Role/title and department
- Why they should be included
- Priority (Primary/Secondary)

Project: {project_description}
""")

chain = stakeholder_prompt | llm
response = chain.invoke({"project_description": project_description})
print(response.content)

## 2. Requirements Elicitation and Extraction (§5.1)
Next, we extract structured requirements from a raw meeting transcript using the EARS (Easy Approach to Requirements Syntax) pattern.

In [ ]:
# Load the mock transcript
with open("data/stakeholder_transcript.txt", "r") as f:
    transcript = f.read()

extraction_prompt = PromptTemplate.from_template("""
Analyze the following stakeholder meeting transcript and extract 5 distinct requirements. 
Format each requirement using the EARS pattern (The system shall... / When <trigger>, the system shall...).

For each requirement, provide:
1. ID (e.g., REQ-01)
2. EARS Statement
3. Type (Functional/Non-Functional/Compliance)
4. Source Speaker

Transcript:
{transcript}
""")

chain = extraction_prompt | llm
extracted_reqs = chain.invoke({"transcript": transcript})
print(extracted_reqs.content)

## 3. User Story Generation (CPFC Pattern) (§5.2)
We take one of the extracted requirements (AML checks) and generate a sprint-ready user story using the Context-Persona-Feature-Criteria (CPFC) pattern.

In [ ]:
cpfc_prompt = PromptTemplate.from_template("""
Convert this requirement into a rigorous agile User Story.

Requirement: The system shall screen the company name and UBOs against the latest OFAC and UN sanctions lists automatically.

Format:
- User story in "As a [persona], I want [feature], so that [benefit]"
- 3 Acceptance criteria using Given/When/Then syntax
- 2 Edge cases
- 1 Non-functional requirement
""")

chain = cpfc_prompt | llm
story = chain.invoke({})
print(story.content)

## 4. Specification Analysis & Gap Detection (§5.3)
AI excels at finding ambiguities or contradictions in specifications before they reach development.

In [ ]:
draft_specs = """
FR-001: The system shall complete the entire onboarding process in under 10 minutes.
FR-002: The system shall verify the identity of all board members and shareholders with >25% equity.
FR-003: If a shareholder is a foreign trust, the compliance officer must manually review the trust deed before approval.
"""

analysis_prompt = PromptTemplate.from_template("""
Act as a business analyst. Review the following requirements and identify:
1. Contradictions between requirements
2. Ambiguities (undefined terms, vague conditions)
3. Missing edge cases

Requirements:
{specs}
""")

chain = analysis_prompt | llm
analysis = chain.invoke({"specs": draft_specs})
print(analysis.content)

## 5. Epic Decomposition (§5.5)
Finally, we use AI to break down a large Epic into a vertical slice of Must-Have, Should-Have, and Nice-to-Have stories.

In [ ]:
epic_description = """
Epic: Corporate Entity Resolution Module
We need to allow corporate clients to upload their ownership documents. The system should 
extract the names, find the ultimate beneficial owners (UBOs), build a visual graph of 
the ownership tree, and flag if any entities are located in offshore tax havens.
"""

decomposition_prompt = PromptTemplate.from_template("""
Decompose the following Epic into 5-7 user stories.

For each story, provide:
- Title
- One-sentence description
- Priority (Must-Have MVP / Should-Have / Nice-to-Have)
- Dependencies (which story must be done first)

Epic: {epic}
""")

chain = decomposition_prompt | llm
decomposition = chain.invoke({"epic": epic_description})
print(decomposition.content)